# 🤖 Simple RAG Demo — HR Policy Assistant

**RAG** stands for **Retrieval-Augmented Generation**. In plain English:

1. We take a document (here, an HR policy handbook).
2. We chop it into small pieces and turn each piece into a list of numbers (an "embedding") that captures its meaning.
3. We store those pieces in a searchable database (a "vector store").
4. When a user asks a question, we **search** the database for the most relevant pieces, and hand them to an AI model to **generate** a final answer.

That's it. No magic — just search + a language model.

**What we use in this notebook:**
- 📄 Data: `data/hr_policy.txt` (a sample HR policy document)
- 🔢 Embeddings: **Jina AI**
- 🗄️ Vector store: **FAISS**
- 🧠 LLM: **Groq** (fast + free-tier friendly)
- 🕸️ Framework: **LangChain** (`create_agent`)

> Before running: make sure the notebook's kernel is set to the `ragenv` virtual environment (top-right corner of VS Code / Jupyter), and that a `.env` file with `GROQ_API_KEY` and `JINA_API_KEY` exists in this folder.

## Step 1 — Import everything we need

We import all the tools upfront so it's clear what's being used and where it comes from.

In [ ]:
import os 
from dotenv import load_dotenv
from pathlib import Path

# langchain
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_community.embeddings import JinaEmbeddings



In [2]:
load_dotenv()

True

In [3]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

print("ENV VAR LOADED")

ENV VAR LOADED


LOADING OUR DATA

In [12]:
BASE_DIR = Path.cwd().parent
print(f"BASE_DIR: {BASE_DIR}")

BASE_DIR: d:\Workshop\practice-rag


In [18]:
DATA_FILE_PATH = BASE_DIR / "data" / "basic-rag" / "hr_policy.txt"
print(f"DATA_FILE_PATH: {DATA_FILE_PATH}")

DATA_FILE_PATH: d:\Workshop\practice-rag\data\basic-rag\hr_policy.txt


#### DATA INGESTION 

In [19]:
loader = TextLoader(DATA_FILE_PATH, encoding="utf-8")
documents = loader.load()

print(f"Loaded file: {DATA_FILE_PATH}")
print(f"Number of documents loaded: {len(documents)}")
print(f"Total characters in document: {len(documents[0].page_content)}")
print("\n--- Preview of first 300 characters ---")
print(documents[0].page_content[:300])

Loaded file: d:\Workshop\practice-rag\data\basic-rag\hr_policy.txt
Number of documents loaded: 1
Total characters in document: 2597

--- Preview of first 300 characters ---
COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carr


#### LANGCHAIN DOCUMENT 

Langchain processes everything in form of documents 


DOCUMENTS : 

PAGE CONTENT -- the actual data 

METADATA  - extra information about the data 

In [20]:
len(documents)

1

In [21]:
print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of e

In [22]:
print(documents[0].metadata)

{'source': 'd:\\Workshop\\practice-rag\\data\\basic-rag\\hr_policy.txt'}


In [23]:
print(f"Total characters in document: {len(documents[0].page_content)}")

Total characters in document: 2597


In [26]:
print(f"Document ID: {documents[0].id}")

Document ID: None


SPLITTING OUR DATA

In [45]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(f"Number of chunks created: {len(chunks)}")

Number of chunks created: 9


In [46]:
# Print preview of first 2 chunks
for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(f"Chunk ID: {chunk.id}")
    print(f"Chunk Metadata: {chunk.metadata}")
    print(f"Chunk Content Preview: {chunk.page_content}")


--- Chunk 1 ---
Chunk ID: None
Chunk Metadata: {'source': 'd:\\Workshop\\practice-rag\\data\\basic-rag\\hr_policy.txt'}
Chunk Content Preview: COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

--- Chunk 2 ---
Chunk ID: None
Chunk Metadata: {'source': 'd:\\Workshop\\practice-rag\\data\\basic-rag\\hr_policy.txt'}
Chunk Content Preview: 1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

--- Chunk 3 ---
Chunk ID: None
Chunk Metadata: {'source': 'd:\\Workshop\\practice-rag\\data\\basic-rag\\hr_policy.txt'}
Chunk Content Preview: 2. WORK FROM HOME POLICY
Employees may work

NOW EACH SPILTED CHUNK IS A DOCUMENT - page content and metadata

In [47]:
print(chunks[0])

page_content='COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)' metadata={'source': 'd:\\Workshop\\practice-rag\\data\\basic-rag\\hr_policy.txt'}


In [48]:
print(chunks[5])

page_content='5. REIMBURSEMENT POLICY
Employees can claim reimbursement for approved business expenses such as travel,
client meals, and internet bills used for official work.
All reimbursement claims must be submitted with valid bills within 30 days of the expense.
Claims are processed within 10 working days after approval from the reporting manager.' metadata={'source': 'd:\\Workshop\\practice-rag\\data\\basic-rag\\hr_policy.txt'}


In [49]:
print(chunks[8].page_content)

8. EXIT POLICY
Upon resignation or termination, employees must complete a clearance process involving
IT, Finance, and HR departments before their last working day.
Full and final settlement, including any pending reimbursements and leave encashment,
is processed within 45 days of the last working day.


In [50]:
print(chunks[7].page_content)

7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.


In [51]:
print(chunks[6].page_content)

6. CODE OF CONDUCT
Employees are expected to maintain professionalism and respect in the workplace.
Harassment, discrimination, or any form of workplace misconduct will not be tolerated
and may result in disciplinary action, including termination.
All employees must complete an annual Code of Conduct training.


EMBEDD OUR DATA

In [55]:
from langchain_community.embeddings import JinaEmbeddings

embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")

print("EMB MODEL READY THE NAME IS ", embeddings_model.model_name)

EMB MODEL READY THE NAME IS  jina-embeddings-v2-base-en


### STORE DATA IN VECTOR DB

In [56]:
from langchain_community.vectorstores import FAISS 

vector_store = FAISS.from_documents(chunks , embeddings_model)

print("CHUNKS ARE STORED" , vector_store.index.ntotal)

CHUNKS ARE STORED 9


WE NEVER STORED IT 

In [57]:
test_query = "How many sick leaves employees get"

## SIMILARITY SEARCH 

top_matches = vector_store.similarity_search(test_query , k=2)
print(f"Query: {test_query}\n")
for i,match in enumerate(top_matches,start=1):
    print(f"--- Match {i} ---")
    print(match.page_content)
    print()


Query: How many sick leaves employees get

--- Match 1 ---
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

--- Match 2 ---
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



#### TOOL

In [58]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})  # returns top 3 relevant chunks

def search_hr_policy(question:str)->str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

### DATA RETRIVAL

LLM 

In [59]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature=0.5  # creativity 
)

llm.model_name

'openai/gpt-oss-120b'

In [60]:
tr = llm.invoke("Hey what is the leave policy")

In [62]:
print(tr.content)

Sure thing! The specifics of a leave policy can vary a lot depending on the company, country, and even the employee’s role or length of service. Below is a **general overview of the most common types of leave** that many organizations include in their policies. If you let me know the name of the company or the jurisdiction you’re interested in, I can tailor the answer more precisely.

---

## 1. Annual / Vacation Leave
| Feature | Typical Details |
|---------|-----------------|
| **Accrual** | Often earned monthly (e.g., 1.25 days per month for a 15‑day annual entitlement). |
| **Carry‑over** | Some companies allow a limited number of unused days to roll over to the next year (e.g., up to 5 days). |
| **Maximum** | Usually 10‑30 days per year, depending on seniority and local labor law. |
| **Request Process** | Submit a request through HR or an internal portal, usually 2‑4 weeks in advance. |

---

## 2. Sick Leave / Medical Leave
| Feature | Typical Details |
|---------|-------------

AI AGENT

3 -- 

LLM - BRAIN 

TOOL - SUPER POWER 

MEMORY - no memory 

In [63]:
from langchain.agents import create_agent      

In [86]:
hr_assistant = create_agent(
    model = llm,
    tools=[search_hr_policy],
    system_prompt= """ 
    
    You are a friendly HR assistant working for Acme Corp. 
    Always use the search_hr_policy tool to look up 
    facts before answering. 
    If the answer isn't in the search results, say you don't know "
    instead of guessing."
    """
)

print("HR assistant agent is ready to answer questions!")

HR assistant agent is ready to answer questions!


In [75]:
def ask_hr_assistant(question: str) -> str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("=" * 60)
    print("QUESTION:", question)
    print("-" * 60)

    response = hr_assistant.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content

    print("ANSWER:", answer)
    print("=" * 60)
    print()
    return answer

In [76]:
response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content": "tell me which org you work for"
            }
        ]
    }
)


In [77]:
response 

{'messages': [HumanMessage(content='tell me which org you work for', additional_kwargs={}, response_metadata={}, id='08280340-c8f6-4d42-af5f-4bc69e1f0c66'),
  AIMessage(content='I’m the friendly HR assistant here at **Acme\u202fCorp**. How can I help you today?', additional_kwargs={'reasoning_content': 'The user asks: "tell me which org you work for". According to developer instructions, we must be a friendly HR assistant working for Acme Corp. Should answer that we work for Acme Corp. No need to search policy. It\'s a factual statement about us. So answer: I work for Acme Corp.'}, response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 208, 'total_tokens': 304, 'completion_time': 0.203613738, 'completion_tokens_details': {'reasoning_tokens': 65}, 'prompt_time': 0.030097817, 'prompt_tokens_details': None, 'queue_time': 0.315679601, 'total_time': 0.233711555}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c800245357', 'service_tier': 'on_demand', 'f

SYSTEM MESSAGE - HR ASSISNT

HUMAN MESSAGE - TELL ME ABOUT POLICIES 

AI MESSAGE  - HEY THESE ARE THEPLOICES 


In [90]:
response["messages"][-2].content

'1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n7. HOLIDAYS\nThe company observes 12 public holidays every year, as per the official holiday calendar\npublished by HR at the start of each year.\nEmployees working on a public holiday are eligible for compensatory leave.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDuring probation, employees are not eligible for paid leave, but may take unpaid leave\nin case of emergencies, subject to manager approval.\nPerformance is reviewed at the end of the probation period to con

In [91]:
print(response["messages"][-1].content)

Employees are entitled to **10 paid sick days per year**. (The HR policy states that “Sick leave is separate from annual leave, and employees get 10 paid sick days per year.”)


In [92]:
for i in range(len(response["messages"])):
    print(f"{response['messages'][i].type} message: {response['messages'][i].content}")

human message: How many sick leaves employees get
ai message: 
tool message: 1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of emergencies, subject to manager approval.
Perfo

In [93]:
question = "How many sick leaves employees get"

response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content": question
            }
        ]
    }
)

In [94]:
response

{'messages': [HumanMessage(content='How many sick leaves employees get', additional_kwargs={}, response_metadata={}, id='d6dd5a82-f45e-4e50-890e-77caab716898'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use search_hr_policy tool to find answer.', 'tool_calls': [{'id': 'fc_5ff263b8-1563-45ce-a489-04dd272ba7b7', 'function': {'arguments': '{"question":"How many sick leaves employees get"}', 'name': 'search_hr_policy'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 207, 'total_tokens': 253, 'completion_time': 0.095786736, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.010892542, 'prompt_tokens_details': None, 'queue_time': 0.309183288, 'total_time': 0.106679278}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_96d96a151c', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0725a-22bc-7683-b92e-2d

In [95]:
for i in range(len(response["messages"])):
    print(f"{response['messages'][i].type} message: {response['messages'][i].content}")

human message: How many sick leaves employees get
ai message: 
tool message: 1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of emergencies, subject to manager approval.
Perfo